In [42]:
### code from sagemath github for regular jdt 
### Annika Christiansen added some code for K-jdt


from sage.arith.misc import factorial
from sage.categories.finite_enumerated_sets import FiniteEnumeratedSets
from sage.categories.infinite_enumerated_sets import InfiniteEnumeratedSets
from sage.categories.sets_cat import Sets
from sage.combinat.integer_vector import IntegerVectors
from sage.combinat.partition import Partition
from sage.combinat.skew_partition import SkewPartition, SkewPartitions
from sage.combinat.tableau import (
    SemistandardTableau,
    StandardTableau,
    Tableau,
    Tableaux,
)
from sage.combinat.words.words import Words
from sage.misc.inherit_comparison import InheritComparisonClasscallMetaclass
from sage.misc.lazy_import import lazy_import
from sage.rings.infinity import PlusInfinity
from sage.rings.integer import Integer
from sage.rings.integer_ring import ZZ
from sage.rings.rational_field import QQ
from sage.structure.list_clone import ClonableList
from sage.structure.parent import Parent
from sage.structure.unique_representation import UniqueRepresentation

lazy_import('sage.matrix.special', 'zero_matrix')
lazy_import('sage.groups.perm_gps.permgroup', 'PermutationGroup')


class SkewTableau(ClonableList,
                  metaclass=InheritComparisonClasscallMetaclass):
    r"""
    A skew tableau.

    Note that Sage by default uses the English convention for partitions and
    tableaux. To change this, see :meth:`Tableaux.options`.

    EXAMPLES::

         sage: st = SkewTableau([[None, 1],[2,3]]); st
         [[None, 1], [2, 3]]
         sage: st.inner_shape()
         [1]
         sage: st.outer_shape()
         [2, 2]

    The ``expr`` form of a skew tableau consists of the inner partition
    followed by a list of the entries in each row from bottom to top::

        sage: SkewTableau(expr=[[1,1],[[5],[3,4],[1,2]]])
        [[None, 1, 2], [None, 3, 4], [5]]

    The ``chain`` form of a skew tableau consists of a list of
    partitions `\lambda_1,\lambda_2,\ldots,`, such that all cells in
    `\lambda_{i+1}` that are not in `\lambda_i` have entry `i`::

        sage: SkewTableau(chain=[[2], [2, 1], [3, 1], [4, 3, 2, 1]])
        [[None, None, 2, 3], [1, 3, 3], [3, 3], [3]]
    """
    @staticmethod
    def __classcall_private__(cls, st=None, expr=None, chain=None):
        """
        Return the skew tableau object corresponding to ``st``.

        EXAMPLES::

            sage: SkewTableau([[None,1],[2,3]])
            [[None, 1], [2, 3]]
            sage: SkewTableau(expr=[[1,1],[[5],[3,4],[1,2]]])
            [[None, 1, 2], [None, 3, 4], [5]]
        """
        if isinstance(st, cls):
            return st
        if expr is not None:
            return SkewTableaux().from_expr(expr)
        if chain is not None:
            return SkewTableaux().from_chain(chain)

        return SkewTableaux()(st)

    def __init__(self, parent, st):
        """
        TESTS::

            sage: st = SkewTableau([[None, 1],[2,3]])
            sage: st = SkewTableau([[None,1,1],[None,2],[4]])
            sage: TestSuite(st).run()

        A skew tableau is immutable, see :issue:`15862`::

            sage: T = SkewTableau([[None,2],[2]])
            sage: t0 = T[0]
            sage: t0[1] = 3
            Traceback (most recent call last):
            ...
            TypeError: 'tuple' object does not support item assignment
            sage: T[0][1] = 5
            Traceback (most recent call last):
            ...
            TypeError: 'tuple' object does not support item assignment
        """
        try:
            st = [tuple(t) for t in st]
        except TypeError:
            raise TypeError("each element of the skew tableau must be an iterable")

        ClonableList.__init__(self, parent, st)

    def __eq__(self, other):
        r"""
        Check whether ``self`` is equal to ``other``.

        .. TODO::

            This overwrites the equality check of
            :class:`~sage.structure.list_clone.ClonableList`
            in order to circumvent the coercion framework.
            Eventually this should be solved more elegantly,
            for example along the lines of what was done for
            `k`-tableaux.

            For now, two elements are equal if their underlying
            defining lists compare equal.

        INPUT:

        - ``other`` -- the element that ``self`` is compared to

        OUTPUT: boolean

        TESTS::

            sage: t = SkewTableau([[None,1,2]])
            sage: t == 0
            False
            sage: t == SkewTableaux()([[None,1,2]])
            True
            sage: t == [(None,1,2)]
            True
            sage: t == [[None,1,2]]
            True

            sage: s = SkewTableau([[1,2]])
            sage: s == 0
            False
            sage: s == Tableau([[1,2]])
            True
        """
        if isinstance(other, (Tableau, SkewTableau)):
            return list(self) == list(other)
        return list(self) == other or [list(row) for row in self] == other

    def __ne__(self, other):
        r"""
        Check whether ``self`` is unequal to ``other``.

        See the documentation of :meth:`__eq__`.

        INPUT:

        - ``other`` -- the element that ``self`` is compared to

        OUTPUT: boolean

        TESTS::

            sage: t = SkewTableau([[None,1,2]])
            sage: t != []
            True
        """
        return not (self == other)

    def __hash__(self):
        """
        Return the hash of ``self``.

        EXAMPLES:

        Check that :issue:`35137` is fixed::

            sage: t = SkewTableau([[None,1,2]])
            sage: hash(t) == hash(tuple(t))
            True
        """
        return hash(tuple(self))

    def check(self):
        r"""
        Check that ``self`` is a valid skew tableau. This is currently far too
        liberal, and only checks some trivial things.

        EXAMPLES::

            sage: t = SkewTableau([[None,1,1],[2]])
            sage: t.check()

            sage: t = SkewTableau([[None, None, 1], [2, 4], [], [3, 4, 5]])
            Traceback (most recent call last):
            ...
            TypeError: a skew tableau cannot have an empty list for a row

            sage: s = SkewTableau([[1, None, None],[2, None],[3]])
            Traceback (most recent call last):
            ...
            TypeError: not a valid skew tableau
        """
        for row in self:
            if not row:
                raise TypeError("a skew tableau cannot have an empty list for a row")
            inside = False
            for x in row:
                if x is not None:
                    inside = True
                elif inside:
                    raise TypeError('not a valid skew tableau')

    def _repr_(self):
        """
        Return a string representation of ``self``.

        For more on the display options, see
        :obj:`SkewTableaux.options`.

        EXAMPLES::

            sage: SkewTableau([[None,2,3],[None,4],[5]])
            [[None, 2, 3], [None, 4], [5]]
        """
        return self.parent().options._dispatch(self, '_repr_', 'display')

    def _repr_list(self):
        """
        Return a string representation of ``self`` as a list of lists.

        EXAMPLES::

            sage: print(SkewTableau([[None,2,3],[None,4],[5]])._repr_list())
            [[None, 2, 3], [None, 4], [5]]
        """
        return repr(self.to_list())

    # See #18024. CombinatorialObject provided __str__, though ClonableList
    # doesn't. Emulate the old functionality. Possibly remove when
    # CombinatorialObject is removed.
    __str__ = _repr_list

    def _repr_diagram(self):
        """
        Return a string representation of ``self`` as a diagram.

        EXAMPLES::

            sage: print(SkewTableau([[None,2,3],[None,4],[5]])._repr_diagram())
              .  2  3
              .  4
              5
        """
        def none_str(x):
            return "  ." if x is None else "%3s" % str(x)
        if self.parent().options('convention') == "French":
            new_rows = ["".join(map(none_str, row)) for row in reversed(self)]
        else:
            new_rows = ["".join(map(none_str, row)) for row in self]
        return '\n'.join(new_rows)

    def _repr_compact(self):
        """
        Return a compact string representation of ``self``.

        EXAMPLES::

            sage: SkewTableau([[None,None,3],[4,5]])._repr_compact()
            '.,.,3/4,5'
            sage: Tableau([])._repr_compact()
            '-'
        """
        if not self:
            return '-'

        def str_rep(x):
            return '%s' % x if x is not None else '.'
        return '/'.join(','.join(str_rep(r) for r in row) for row in self)

    def pp(self):
        """
        Return a pretty print string of the tableau.

        EXAMPLES::

            sage: SkewTableau([[None,2,3],[None,4],[5]]).pp()
              .  2  3
              .  4
              5
        """
        print(self._repr_diagram())

    def _ascii_art_(self):
        r"""
        Return an ascii art representation of ``self``.

        TESTS::

            sage: T1 = SkewTableau([[None,2,3],[None,4],[5]])
            sage: T2 = SkewTableau([[None,None,3],[4,5]])
            sage: ascii_art([T1, T2])
            [   .  2  3            ]
            [   .  4       .  .  3 ]
            [   5      ,   4  5    ]
        """
        from sage.typeset.ascii_art import AsciiArt
        return AsciiArt(self._repr_diagram().splitlines())

    def _unicode_art_(self):
        """
        Return a unicode art representation of ``self``.

        TESTS::

            sage: T = SkewTableau([[None,None,1,1,2],[None,2,3],[None,4],[None,5],[6]])
            sage: unicode_art(T)
                    ┌───┬───┬───┐
                    │ 1 │ 1 │ 2 │
                ┌───┼───┼───┴───┘
                │ 2 │ 3 │
                ├───┼───┘
                │ 4 │
                ├───┤
                │ 5 │
            ┌───┼───┘
            │ 6 │
            └───┘
        """
        from sage.combinat.output import ascii_art_table
        from sage.typeset.unicode_art import UnicodeArt
        return UnicodeArt(ascii_art_table(self, use_unicode=True).splitlines())

    def _latex_(self):
        r"""
        Return a `\LaTeX` representation of ``self``.

        EXAMPLES::

            sage: latex(SkewTableau([[None,2,3],[None,4],[5]]))
            {\def\lr#1{\multicolumn{1}{|@{\hspace{.6ex}}c@{\hspace{.6ex}}|}{\raisebox{-.3ex}{$#1$}}}
            \raisebox{-.6ex}{$\begin{array}[b]{*{3}c}\cline{2-3}
            &\lr{2}&\lr{3}\\\cline{2-3}
            &\lr{4}\\\cline{1-2}
            \lr{5}\\\cline{1-1}
            \end{array}$}
            }
        """
        from sage.combinat.output import tex_from_array
        return tex_from_array(self)

    def outer_shape(self):
        """
        Return the outer shape of ``self``.

        EXAMPLES::

            sage: SkewTableau([[None,1,2],[None,3],[4]]).outer_shape()
            [3, 2, 1]
        """
        return Partition([len(row) for row in self])

    def inner_shape(self):
        """
        Return the inner shape of ``self``.

        EXAMPLES::

            sage: SkewTableau([[None,1,2],[None,3],[4]]).inner_shape()
            [1, 1]
            sage: SkewTableau([[1,2],[3,4],[7]]).inner_shape()
            []
            sage: SkewTableau([[None,None,None,2,3],[None,1],[None],[2]]).inner_shape()
            [3, 1, 1]
        """
        return Partition([x for x in (row.count(None) for row in self) if x != 0])

    def shape(self):
        r"""
        Return the shape of ``self``.

        EXAMPLES::

            sage: SkewTableau([[None,1,2],[None,3],[4]]).shape()
            [3, 2, 1] / [1, 1]
        """
        return SkewPartition([self.outer_shape(), self.inner_shape()])

    def outer_size(self):
        """
        Return the size of the outer shape of ``self``.

        EXAMPLES::

            sage: SkewTableau([[None, 2, 4], [None, 3], [1]]).outer_size()
            6
            sage: SkewTableau([[None, 2], [1, 3]]).outer_size()
            4
        """
        return self.outer_shape().size()

    def inner_size(self):
        """
        Return the size of the inner shape of ``self``.

        EXAMPLES::

            sage: SkewTableau([[None, 2, 4], [None, 3], [1]]).inner_size()
            2
            sage: SkewTableau([[None, 2], [1, 3]]).inner_size()
            1
        """
        return self.inner_shape().size()

    def size(self):
        """
        Return the number of cells in ``self``.

        EXAMPLES::

            sage: SkewTableau([[None, 2, 4], [None, 3], [1]]).size()
            4
            sage: SkewTableau([[None, 2], [1, 3]]).size()
            3
        """
        one = ZZ.one()
        return sum(one for row in self for x in row if x is not None)

    def conjugate(self):
        """
        Return the conjugate of ``self``.

        EXAMPLES::

            sage: SkewTableau([[None,1],[2,3]]).conjugate()
            [[None, 2], [1, 3]]
        """
        conj_shape = self.outer_shape().conjugate()

        conj = [[None] * row_length for row_length in conj_shape]

        for i in range(len(conj)):
            for j in range(len(conj[i])):
                conj[i][j] = self[j][i]

        return SkewTableau(conj)

    def to_word_by_row(self):
        """
        Return a word obtained from a row reading of ``self``.

        This is the word obtained by concatenating the rows from
        the bottommost one (in English notation) to the topmost one.

        EXAMPLES::

            sage: s = SkewTableau([[None,1],[2,3]])
            sage: s.pp()
              .  1
              2  3
            sage: s.to_word_by_row()
            word: 231
            sage: s = SkewTableau([[None, 2, 4], [None, 3], [1]])
            sage: s.pp()
              .  2  4
              .  3
              1
            sage: s.to_word_by_row()
            word: 1324

        TESTS::

            sage: SkewTableau([[None, None, None], [None]]).to_word_by_row()
            word:
            sage: SkewTableau([]).to_word_by_row()
            word:
        """
        word = [x for row in reversed(self) for x in row if x is not None]
        return Words("positive integers")(word)

    def to_word_by_column(self):
        """
        Return the word obtained from a column reading of the skew
        tableau.

        This is the word obtained by concatenating the columns from
        the rightmost one (in English notation) to the leftmost one.

        EXAMPLES::

            sage: s = SkewTableau([[None,1],[2,3]])
            sage: s.pp()
              .  1
              2  3
            sage: s.to_word_by_column()
            word: 132

        ::

            sage: s = SkewTableau([[None, 2, 4], [None, 3], [1]])
            sage: s.pp()
            .  2  4
            .  3
            1
            sage: s.to_word_by_column()
            word: 4231
        """
        return self.conjugate().to_word_by_row()

    to_word = to_word_by_row

    def to_permutation(self):
        """
        Return a permutation with the entries of ``self`` obtained by reading
        ``self`` row by row, from the bottommost to the topmost row, with
        each row being read from left to right, in English convention.
        See :meth:`to_word_by_row()`.

        EXAMPLES::

            sage: SkewTableau([[None,2],[3,4],[None],[1]]).to_permutation()
            [1, 3, 4, 2]
            sage: SkewTableau([[None,2],[None,4],[1],[3]]).to_permutation()
            [3, 1, 4, 2]
            sage: SkewTableau([[None]]).to_permutation()
            []
        """
        from sage.combinat.permutation import Permutation
        perm = [i for row in reversed(self) for i in row if i is not None]
        return Permutation(perm)

    def weight(self):
        r"""
        Return the weight (aka evaluation) of the tableau ``self``.
        Trailing zeroes are omitted when returning the weight.

        The weight of a skew tableau `T` is the sequence
        `(a_1, a_2, a_3, \ldots )`, where `a_k` is the number of
        entries of `T` equal to `k`. This sequence contains only
        finitely many nonzero entries.

        The weight of a skew tableau `T` is the same as the weight
        of the reading word of `T`, for any reading order.

        :meth:`evaluation` is a synonym for this method.

        EXAMPLES::

            sage: SkewTableau([[1,2],[3,4]]).weight()
            [1, 1, 1, 1]

            sage: SkewTableau([[None,2],[None,4],[None,5],[None]]).weight()
            [0, 1, 0, 1, 1]

            sage: SkewTableau([]).weight()
            []

            sage: SkewTableau([[None,None,None],[None]]).weight()
            []

            sage: SkewTableau([[None,3,4],[None,6,7],[4,8],[5,13],[6],[7]]).weight()
            [0, 0, 1, 2, 1, 2, 2, 1, 0, 0, 0, 0, 1]

        TESTS:

        We check that this agrees with going to the word::

            sage: t = SkewTableau([[None,None,4,7,15],[6,2,16],[2,3,19],[4,5],[7]])
            sage: def by_word(T):
            ....:     ed = T.to_word().evaluation_dict()
            ....:     m = max(ed) + 1
            ....:     return [ed.get(k, 0) for k in range(1, m)]
            sage: by_word(t) == t.weight()
            True
            sage: SST = SemistandardTableaux(shape=[3,1,1])
            sage: all(by_word(t) == SkewTableau(t).weight() for t in SST)               # needs sage.modules
            True
        """
        if (not self) or all(c is None for row in self for c in row):
            return []
        m = max(c for row in self for c in row if c is not None)
        if m is None:
            return []
        res = [0] * m
        for row in self:
            for i in row:
                if (i is not None) and i > 0:
                    res[i - 1] += 1
        return res

    evaluation = weight

    def is_standard(self) -> bool:
        """
        Return ``True`` if ``self`` is a standard skew tableau and ``False``
        otherwise.

        EXAMPLES::

            sage: SkewTableau([[None, 2], [1, 3]]).is_standard()
            True
            sage: SkewTableau([[None, 2], [2, 4]]).is_standard()
            False
            sage: SkewTableau([[None, 3], [2, 4]]).is_standard()
            False
            sage: SkewTableau([[None, 2], [2, 4]]).is_standard()
            False
        """
        # Check to make sure that it is filled with 1...size
        w = [i for row in self for i in row if i is not None]
        return sorted(w) == list(range(1, len(w) + 1)) and self.is_semistandard()

    def is_semistandard(self) -> bool:
        """
        Return ``True`` if ``self`` is a semistandard skew tableau and
        ``False`` otherwise.

        EXAMPLES::

            sage: SkewTableau([[None, 2, 2], [1, 3]]).is_semistandard()
            True
            sage: SkewTableau([[None, 2], [2, 4]]).is_semistandard()
            True
            sage: SkewTableau([[None, 3], [2, 4]]).is_semistandard()
            True
            sage: SkewTableau([[None, 2], [1, 2]]).is_semistandard()
            False
            sage: SkewTableau([[None, 2, 3]]).is_semistandard()
            True
            sage: SkewTableau([[None, 3, 2]]).is_semistandard()
            False
            sage: SkewTableau([[None, 2, 3], [1, 4]]).is_semistandard()
            True
            sage: SkewTableau([[None, 2, 3], [1, 2]]).is_semistandard()
            False
            sage: SkewTableau([[None, 2, 3], [None, None, 4]]).is_semistandard()
            False
        """
        if not self:
            return True

        # Is it weakly increasing along the rows?
        for row in self:
            if any(row[c] is not None and row[c] > row[c + 1]
                   for c in range(len(row) - 1)):
                return False

        # Is it strictly increasing down columns?
        for row, next in zip(self, self[1:]):
            if any(row[c] is not None and row[c] >= next[c] for c in range(len(next))):
                return False

        return True

    def to_tableau(self):
        """
        Return a tableau with the same filling. This only works if the
        inner shape of the skew tableau has size zero.

        EXAMPLES::

            sage: SkewTableau([[1,2],[3,4]]).to_tableau()
            [[1, 2], [3, 4]]
        """

        if self.inner_size() != 0:
            raise ValueError("the inner size of the skew tableau must be 0")
        from sage.combinat.tableau import Tableau
        return Tableau(self[:])

    def restrict(self, n):
        """
        Return the restriction of the (semi)standard skew tableau to all
        the numbers less than or equal to ``n``.

        .. NOTE::

            If only the outer shape of the restriction, rather than
            the whole restriction, is needed, then the faster method
            :meth:`restriction_outer_shape` is preferred. Similarly if
            only the skew shape is needed, use :meth:`restriction_shape`.

        EXAMPLES::

            sage: SkewTableau([[None,1],[2],[3]]).restrict(2)
            [[None, 1], [2]]
            sage: SkewTableau([[None,1],[2],[3]]).restrict(1)
            [[None, 1]]
            sage: SkewTableau([[None,1],[1],[2]]).restrict(1)
            [[None, 1], [1]]
        """
        data = ([y for y in x if y is None or y <= n] for x in self)
        return SkewTableau([z for z in data if z])

    def restriction_outer_shape(self, n):
        """
        Return the outer shape of the restriction of the semistandard skew
        tableau ``self`` to `n`.

        If `T` is a semistandard skew tableau and `n` is a nonnegative
        integer, then the restriction of `T` to `n` is defined as the
        (semistandard) skew tableau obtained by removing all cells filled
        with entries greater than `n` from `T`.

        This method computes merely the outer shape of the restriction.
        For the restriction itself, use :meth:`restrict`.

        EXAMPLES::

            sage: SkewTableau([[None,None],[2,3],[3,4]]).restriction_outer_shape(3)
            [2, 2, 1]
            sage: SkewTableau([[None,2],[None],[4],[5]]).restriction_outer_shape(2)
            [2, 1]
            sage: T = SkewTableau([[None,None,3,5],[None,4,4],[17]])
            sage: T.restriction_outer_shape(0)
            [2, 1]
            sage: T.restriction_outer_shape(2)
            [2, 1]
            sage: T.restriction_outer_shape(3)
            [3, 1]
            sage: T.restriction_outer_shape(4)
            [3, 3]
            sage: T.restriction_outer_shape(19)
            [4, 3, 1]
        """
        from sage.combinat.partition import _Partitions
        one = ZZ.one()
        res = [sum(one for y in row if y is None or y <= n) for row in self]
        return _Partitions(res)

    def restriction_shape(self, n):
        """
        Return the skew shape of the restriction of the semistandard
        skew tableau ``self`` to ``n``.

        If `T` is a semistandard skew tableau and `n` is a nonnegative
        integer, then the restriction of `T` to `n` is defined as the
        (semistandard) skew tableau obtained by removing all cells filled
        with entries greater than `n` from `T`.

        This method computes merely the skew shape of the restriction.
        For the restriction itself, use :meth:`restrict`.

        EXAMPLES::

            sage: SkewTableau([[None,None],[2,3],[3,4]]).restriction_shape(3)
            [2, 2, 1] / [2]
            sage: SkewTableau([[None,2],[None],[4],[5]]).restriction_shape(2)
            [2, 1] / [1, 1]
            sage: T = SkewTableau([[None,None,3,5],[None,4,4],[17]])
            sage: T.restriction_shape(0)
            [2, 1] / [2, 1]
            sage: T.restriction_shape(2)
            [2, 1] / [2, 1]
            sage: T.restriction_shape(3)
            [3, 1] / [2, 1]
            sage: T.restriction_shape(4)
            [3, 3] / [2, 1]
        """
        return SkewPartition([self.restriction_outer_shape(n), self.inner_shape()])

    def to_chain(self, max_entry=None):
        r"""
        Return the chain of partitions corresponding to the (semi)standard
        skew tableau ``self``.

        The optional keyword parameter ``max_entry`` can be used to
        customize the length of the chain. Specifically, if this parameter
        is set to a nonnegative integer ``n``, then the chain is
        constructed from the positions of the letters `1, 2, \ldots, n`
        in the tableau.

        EXAMPLES::

            sage: SkewTableau([[None,1],[2],[3]]).to_chain()
            [[1], [2], [2, 1], [2, 1, 1]]
            sage: SkewTableau([[None,1],[1],[2]]).to_chain()
            [[1], [2, 1], [2, 1, 1]]
            sage: SkewTableau([[None,1],[1],[2]]).to_chain(max_entry=2)
            [[1], [2, 1], [2, 1, 1]]
            sage: SkewTableau([[None,1],[1],[2]]).to_chain(max_entry=3)
            [[1], [2, 1], [2, 1, 1], [2, 1, 1]]
            sage: SkewTableau([[None,1],[1],[2]]).to_chain(max_entry=1)
            [[1], [2, 1]]
            sage: SkewTableau([[None,None,2],[None,3],[None,5]]).to_chain(max_entry=6)
            [[2, 1, 1], [2, 1, 1], [3, 1, 1], [3, 2, 1], [3, 2, 1], [3, 2, 2], [3, 2, 2]]
            sage: SkewTableau([]).to_chain()
            [[]]
            sage: SkewTableau([]).to_chain(max_entry=1)
            [[], []]

        TESTS:

        Check that :meth:`to_chain()` does not skip letters::

            sage: t = SkewTableau([[None, 2, 3], [3]])
            sage: t.to_chain()
            [[1], [1], [2], [3, 1]]

            sage: T = SkewTableau([[None]])
            sage: T.to_chain()
            [[1]]
        """
        if max_entry is None:
            if (not self) or all(c is None for row in self for c in row):
                max_entry = 0
            else:
                max_entry = max(c for row in self for c in row if c is not None)
        return [self.restriction_outer_shape(x) for x in range(max_entry + 1)]

    def slide(self, corner=None, return_vacated=False):
        """
        Apply a jeu de taquin slide to ``self`` on the specified inner
        corner and return the resulting tableau.

        If no corner is given, the topmost inner corner is chosen.

        The optional parameter ``return_vacated=True`` causes
        the output to be the pair ``(t, (i, j))`` where ``t`` is the new
        tableau and ``(i, j)`` are the coordinates of the vacated square.

        See [Ful1997]_ p12-13.

        EXAMPLES::

            sage: st = SkewTableau([[None, None, None, None, 2], [None, None, None, None, 6], [None, 2, 4, 4], [2, 3, 6], [5, 5]])
            sage: st.slide((2, 0))
            [[None, None, None, None, 2], [None, None, None, None, 6], [2, 2, 4, 4], [3, 5, 6], [5]]
            sage: st2 = SkewTableau([[None, None, 3], [None, 2, 4], [1, 5]])
            sage: st2.slide((1, 0), True)
            ([[None, None, 3], [1, 2, 4], [5]], (2, 1))

        TESTS::

            sage: st
            [[None, None, None, None, 2], [None, None, None, None, 6],
             [None, 2, 4, 4], [2, 3, 6], [5, 5]]
        """
        new_st = self.to_list()
        inner_corners = self.inner_shape().corners()
        outer_corners = self.outer_shape().corners()
        if corner is not None:
            if tuple(corner) not in inner_corners:
                raise ValueError("corner must be an inner corner")
        else:
            if not inner_corners:
                return self
            corner = inner_corners[0]

        spotl, spotc = corner
        while (spotl, spotc) not in outer_corners:
            # Check to see if there is nothing to the right
            if spotc == len(new_st[spotl]) - 1:
                # Swap the hole with the cell below
                new_st[spotl][spotc] = new_st[spotl + 1][spotc]
                new_st[spotl + 1][spotc] = None
                spotl += 1
                continue

            # Check to see if there is nothing below
            if spotl == len(new_st) - 1 or len(new_st[spotl + 1]) <= spotc:
                # Swap the hole with the cell to the right
                new_st[spotl][spotc] = new_st[spotl][spotc + 1]
                new_st[spotl][spotc + 1] = None
                spotc += 1
                continue

            # If we get to this stage, we need to compare
            below = new_st[spotl + 1][spotc]
            right = new_st[spotl][spotc + 1]
            if below <= right:
                # Swap with the cell below
                new_st[spotl][spotc] = new_st[spotl + 1][spotc]
                new_st[spotl + 1][spotc] = None
                spotl += 1
                continue

            # Otherwise swap with the cell to the right
            new_st[spotl][spotc] = new_st[spotl][spotc + 1]
            new_st[spotl][spotc + 1] = None
            spotc += 1

        # Clean up to remove the "None" at an outside corner
        # Remove the last row if there is nothing left in it
        new_st[spotl].pop()
        if not new_st[spotl]:
            new_st.pop()

        if return_vacated:
            return (SkewTableau(new_st), (spotl, spotc))
        return SkewTableau(new_st)

    def backward_slide(self, corner=None):
        r"""
        Apply a backward jeu de taquin slide on the specified outside
        ``corner`` of ``self``.

        Backward jeu de taquin slides are defined in Section 3.7 of
        [Sag2001]_.

        .. WARNING::

            The :meth:`inner_corners` and :meth:`outer_corners` are the
            :meth:`sage.combinat.partition.Partition.corners` of the inner and
            outer partitions of the skew shape. They are different from the
            inner/outer corners defined in [Sag2001]_.

            The "inner corners" of [Sag2001]_ may be found by calling
            :meth:`outer_corners`. The "outer corners" of [Sag2001]_ may be
            found by calling ``self.outer_shape().outside_corners()``.

        EXAMPLES::

            sage: T = SkewTableaux()([[2, 2], [4, 4], [5]])
            sage: Tableaux.options.display='array'
            sage: Q = T.backward_slide(); Q
            . 2 2
            4 4
            5
            sage: Q.backward_slide((1, 2))
            . 2 2
            . 4 4
            5
            sage: Q.reverse_slide((1, 2)) == Q.backward_slide((1, 2))
            True

            sage: T = SkewTableaux()([[1, 3],[3],[5]]); T
            1 3
            3
            5
            sage: T.reverse_slide((1,1))
            . 1
            3 3
            5

        TESTS::

            sage: T = SkewTableaux()([[2, 2], [4, 4], [5]])
            sage: Q = T.backward_slide((0, 2))
            sage: Q.backward_slide((2,1))
            . 2 2
            . 4
            4 5
            sage: Q.backward_slide((3,0))
            . 2 2
            . 4
            4
            5
            sage: Q = T.backward_slide((2,1)); Q
            . 2
            2 4
            4 5
            sage: Q.backward_slide((3,0))
            . 2
            . 4
            2 5
            4
            sage: Q = T.backward_slide((3,0)); Q
            . 2
            2 4
            4
            5
            sage: Q.backward_slide((4,0))
            . 2
            . 4
            2
            4
            5
            sage: Tableaux.options.display='list'
        """
        new_st = self.to_list()
        inner_outside_corners = self.inner_shape().outside_corners()
        outer_outisde_corners = self.outer_shape().outside_corners()
        if corner is not None:
            if tuple(corner) not in outer_outisde_corners:
                raise ValueError("corner must be an outside corner"
                                 " of the outer shape")
        else:
            if not outer_outisde_corners:
                return self
            corner = outer_outisde_corners[0]

        i, j = corner

        # add the empty cell
        # the column only matters if it is zeroth column, in which
        # case we need to add a new row.
        if not j:
            new_st.append(list())
        new_st[i].append(None)

        while (i, j) not in inner_outside_corners:
            # get the value of the cell above the temporarily empty cell (if
            # it exists)
            if i > 0:
                P_up = new_st[i-1][j]
            else:
                P_up = -1  # a dummy value less than all positive numbers

            # get the value of the cell to the left of the temp. empty cell
            # (if it exists)
            if j > 0:
                P_left = new_st[i][j-1]
            else:
                P_left = -1  # a dummy value less than all positive numbers

            # get the next cell
            # p_left will always be positive, but if P_left
            # and P_up are both None, then it will return
            # -1, which doesn't trigger the conditional
            if P_left > P_up:
                new_st[i][j] = P_left
                j = j - 1
            else:  # if they are equal, we slide up
                new_st[i][j] = P_up
                i = i - 1
        # We don't need to reset the intermediate cells inside the loop
        # because the conditional above will continue to overwrite it until
        # the while loop terminates. We do need to reset it at the end.
        new_st[i][j] = None

        return SkewTableaux()(new_st)

    reverse_slide = backward_slide

    def rectify(self, algorithm=None):
        """
        Return a :class:`StandardTableau`, :class:`SemistandardTableau`,
        or just :class:`Tableau` formed by applying the jeu de taquin
        process to ``self``.

        See page 15 of [Ful1997]_.

        INPUT:

        - ``algorithm`` -- (optional) if set to ``'jdt'``, rectifies by jeu de
          taquin; if set to ``'schensted'``, rectifies by Schensted insertion
          of the reading word. Otherwise, guesses which will be faster.

        EXAMPLES::

            sage: S = SkewTableau([[None,1],[2,3]])
            sage: S.rectify()
            [[1, 3], [2]]
            sage: T = SkewTableau([[None, None, None, 4],[None,None,1,6],[None,None,5],[2,3]])
            sage: T.rectify()
            [[1, 3, 4, 6], [2, 5]]
            sage: T.rectify(algorithm='jdt')
            [[1, 3, 4, 6], [2, 5]]
            sage: T.rectify(algorithm='schensted')
            [[1, 3, 4, 6], [2, 5]]
            sage: T.rectify(algorithm='spaghetti')
            Traceback (most recent call last):
            ...
            ValueError: algorithm must be 'jdt', 'schensted', or None

        TESTS::

            sage: S
            [[None, 1], [2, 3]]
            sage: T
            [[None, None, None, 4], [None, None, 1, 6], [None, None, 5], [2, 3]]
        """
        mu_size = self.inner_shape().size()

        # Roughly, use jdt with a small inner shape, Schensted with a large one
        if algorithm is None:
            la = self.outer_shape()
            la_size = la.size()
            if mu_size ** 2 < len(la) * (la_size - mu_size):
                algorithm = 'jdt'
            else:
                algorithm = 'schensted'

        if algorithm == 'jdt':
            rect = self
            for _ in range(mu_size):
                rect = rect.slide()
        elif algorithm == 'schensted':
            w = [x for row in reversed(self) for x in row
                 if x is not None]
            rect = Tableau([]).insert_word(w)
        else:
            raise ValueError("algorithm must be 'jdt', 'schensted', or None")
        if self in StandardSkewTableaux():
            return StandardTableau(rect[:])
        if self in SemistandardSkewTableaux():
            return SemistandardTableau(rect[:])
        return Tableau(rect)


    def to_list(self):
        r"""
        Return a (mutable) list representation of ``self``.

        EXAMPLES::

            sage: stlist = [[None, None, 3], [None, 1, 3], [2, 2]]
            sage: st = SkewTableau(stlist)
            sage: st.to_list()
            [[None, None, 3], [None, 1, 3], [2, 2]]
            sage: st.to_list() == stlist
            True
        """
        return [list(row) for row in self]



In [43]:

def outer_shape(tab):
    """
    Return the outer shape of ``self``.

    EXAMPLES::

        sage: SkewTableau([[None,1,2],[None,3],[4]]).outer_shape()
        [3, 2, 1]
    """
    return Partition([len(row) for row in tab])

def inner_shape(tab):
    """
    Return the inner shape of ``self``.

    EXAMPLES::

        sage: SkewTableau([[None,1,2],[None,3],[4]]).inner_shape()
        [1, 1]
        sage: SkewTableau([[1,2],[3,4],[7]]).inner_shape()
        []
        sage: SkewTableau([[None,None,None,2,3],[None,1],[None],[2]]).inner_shape()
        [3, 1, 1]
    """
    return Partition([x for x in (row.count(None) for row in tab) if x != 0])

def shape(tab):
    r"""
    Return the shape of ``self``.

    EXAMPLES::

        sage: SkewTableau([[None,1,2],[None,3],[4]]).shape()
        [3, 2, 1] / [1, 1]
    """
    return SkewPartition([outer_shape(tab), inner_shape(tab)])

In [73]:
def Kslide(skewtab, corner = None):
    new_st = skewtab.to_list()
    inner_corners = skewtab.inner_shape().corners()
    outer_corners = skewtab.outer_shape().corners()
    print(f" Starting Tableau {new_st} and it's inner corners {inner_corners} and outer corners {outer_corners}")
    """ maybe edit this with input a tableaux to give K-rect ordering?"""
    if corner is not None:
            if tuple(corner) not in inner_corners:
                raise ValueError("corner must be an inner corner")
    else:
        if not inner_corners:
            return skewtab
        corner = inner_corners[0]

    spotl, spotc = corner
    while (spotl, spotc) not in outer_corners:
        # Check to see if there is nothing to the right
        if spotc == len(new_st[spotl]) - 1:
            #print('case nothing right')
            # Swap with the cell below
            new_st[spotl][spotc] = new_st[spotl + 1][spotc]
            new_st[spotl + 1][spotc] = None
            spotl += 1
            print(f" intermediate tableau {new_st} and the corner {(spotl,spotc)}")
            continue

        # Check to see if there is nothing below
        if spotl == len(new_st) - 1 or len(new_st[spotl + 1]) <= spotc:
            # Swap with the cell to the right
            #print('case nothing below')
            new_st[spotl][spotc] = new_st[spotl][spotc + 1]
            new_st[spotl][spotc + 1] = None
            spotc += 1
            print(f" intermediate tableau {new_st} and the corner {(spotl,spotc)}")
            continue

        #print('case compare')
        below = new_st[spotl + 1][spotc]
        right = new_st[spotl][spotc + 1]
        if below == right:
            #print('k-move')
            #print(f" here's the corner before {(spotl,spotc)}")
            
            # Create two of the cell (one in below & one in right) and the others get merged into original cell loc
            new_st[spotl][spotc] = new_st[spotl + 1][spotc]
            new_st[spotl + 1][spotc] = None
            new_st[spotl][spotc+1] = None
            if (spotl,spotc+1) in outer_corners and (spotl+1,spotc) not in outer_corners:
                spotl += 1
            elif (spotl+1,spotc) in outer_corners and (spotl,spotc+1) not in outer_corners:
                spotc += 1
            elif (spotl+1,spotc) in outer_corners and (spotl,spotc+1) in outer_corners:
                ## arbitrary choice, should end the loop
                spotc += 1
            else:
                new_st[spotl + 1][spotc] = new_st[spotl + 1][spotc+1]
                new_st[spotl][spotc+1] = new_st[spotl + 1][spotc+1]
                new_st[spotl+1][spotc+1] = None
                spotc += 1
                spotl += 1
                
                
            
            print(f" intermediate tableau {new_st} and the corner {(spotl,spotc)}")
            continue
        elif below < right:
            #print('swap below')
            # Swap with the cell below
            new_st[spotl][spotc] = new_st[spotl + 1][spotc]
            new_st[spotl + 1][spotc] = None
            spotl += 1
            print(f" intermediate tableau {new_st}")
            continue
        else:
            # Swap with the cell to the right
            #print('swap right')
            new_st[spotl][spotc] = new_st[spotl][spotc + 1]
            new_st[spotl][spotc + 1] = None
            spotc += 1
            print(f" intermediate tableau {new_st}")
            continue

    final_tab = []
    for row in new_st:
        if row[-1] is None and len(row) > 1:
            final_tab.append(row[:-1])
        elif row[-1] is None and len(row) == 1:
            continue
        else:
            final_tab.append(row)
        #print(final_tab)

    #print(f" here's our final tableaux = {final_tab}")
    return SkewTableau(final_tab)

def Krect_from_slides(tab, rect_order):
    print(f" here's the starting corner to slide {rect_order[0]}")
    for corner in rect_order:
        
        tab_new = Kslide(tab,corner)
        tab = tab_new

    return tab
    

In [74]:
t = SkewTableau([[None,None,3],[None,1],[1]])
order = [(0,1),(1,0),(0,0)]

Krect_from_slides(t, order)

 here's the starting corner to slide (0, 1)
 Starting Tableau [[None, None, 3], [None, 1], [1]] and it's inner corners [(0, 1), (1, 0)] and outer corners [(0, 2), (1, 1), (2, 0)]
 intermediate tableau [[None, 1, 3], [None, None], [1]]
 Starting Tableau [[None, 1, 3], [None], [1]] and it's inner corners [(1, 0)] and outer corners [(0, 2), (2, 0)]
 intermediate tableau [[None, 1, 3], [1], [None]] and the corner (2, 0)
 Starting Tableau [[None, 1, 3], [1]] and it's inner corners [(0, 0)] and outer corners [(0, 2), (1, 0)]
 intermediate tableau [[1, None, 3], [None]] and the corner (0, 1)
 intermediate tableau [[1, 3, None], [None]] and the corner (0, 2)


[[1, 3]]

In [75]:
t = SkewTableau([[None,None,3],[None,1],[1]])
order_rev = [(1,0),(0,1),(0,0)]

Krect_from_slides(t, order_rev)

 here's the starting corner to slide (1, 0)
 Starting Tableau [[None, None, 3], [None, 1], [1]] and it's inner corners [(0, 1), (1, 0)] and outer corners [(0, 2), (1, 1), (2, 0)]
 intermediate tableau [[None, None, 3], [1, None], [None]] and the corner (1, 1)
 Starting Tableau [[None, None, 3], [1]] and it's inner corners [(0, 1)] and outer corners [(0, 2), (1, 0)]
 intermediate tableau [[None, 3, None], [1]] and the corner (0, 2)
 Starting Tableau [[None, 3], [1]] and it's inner corners [(0, 0)] and outer corners [(0, 1), (1, 0)]
 intermediate tableau [[1, 3], [None]]


[[1, 3]]

In [47]:
tab = SkewTableau([[None,None,None,2],[None,None,2],[1,3,4]])
rect_order1 = [(0,2),(1,1),(1,0),(0,1),(0,0)]
rect_order2 = [(1,1),(1,0),(0,2),(0,1),(0,0)]

Krect_from_slides(tab, rect_order1)

 Starting Tableau [[None, None, None, 2], [None, None, 2], [1, 3, 4]] and it's inner corners [(0, 2), (1, 1)] and outer corners [(0, 3), (2, 2)]
 intermediate tableau [[None, None, 2, None], [None, None, None], [1, 3, 4]] and the corner (1, 2)
 intermediate tableau [[None, None, 2, None], [None, None, 4], [1, 3, None]] and the corner (2, 2)
 Starting Tableau [[None, None, 2], [None, None, 4], [1, 3]] and it's inner corners [(1, 1)] and outer corners [(1, 2), (2, 1)]
 intermediate tableau [[None, None, 2], [None, 3, 4], [1, None]]
 Starting Tableau [[None, None, 2], [None, 3, 4], [1]] and it's inner corners [(0, 1), (1, 0)] and outer corners [(1, 2), (2, 0)]
 intermediate tableau [[None, None, 2], [1, 3, 4], [None]]
 Starting Tableau [[None, None, 2], [1, 3, 4]] and it's inner corners [(0, 1)] and outer corners [(1, 2)]
 intermediate tableau [[None, 2, None], [1, 3, 4]]
 intermediate tableau [[None, 2, 4], [1, 3, None]] and the corner (1, 2)
 Starting Tableau [[None, 2, 4], [1, 3]] and 

[[1, 2, 4], [3]]

In [48]:
Krect_from_slides(tab, rect_order2)

 Starting Tableau [[None, None, None, 2], [None, None, 2], [1, 3, 4]] and it's inner corners [(0, 2), (1, 1)] and outer corners [(0, 3), (2, 2)]
 intermediate tableau [[None, None, None, 2], [None, 2, None], [1, 3, 4]]
 intermediate tableau [[None, None, None, 2], [None, 2, 4], [1, 3, None]] and the corner (2, 2)
 Starting Tableau [[None, None, None, 2], [None, 2, 4], [1, 3]] and it's inner corners [(0, 2), (1, 0)] and outer corners [(0, 3), (1, 2), (2, 1)]
 intermediate tableau [[None, None, None, 2], [1, 2, 4], [None, 3]]
 intermediate tableau [[None, None, None, 2], [1, 2, 4], [3, None]] and the corner (2, 1)
 Starting Tableau [[None, None, None, 2], [1, 2, 4], [3]] and it's inner corners [(0, 2)] and outer corners [(0, 3), (1, 2), (2, 0)]
 intermediate tableau [[None, None, 2, None], [1, 2, 4], [3]]
 Starting Tableau [[None, None, 2], [1, 2, 4], [3]] and it's inner corners [(0, 1)] and outer corners [(1, 2), (2, 0)]
 intermediate tableau [[None, 2, 4], [1, 4, None], [3]] and the co

[[1, 2, 4], [3, 4]]

In [49]:
tab = SkewTableau([[None,None,None,2],[None,None,2],[1,3,4]])
one_slide = Kslide(tab,(0,2))
print(one_slide)

two_slide = Kslide(one_slide,(1,1))
print(two_slide)

three_slide = Kslide(two_slide,(1,0))
print(three_slide)

four_slide = Kslide(three_slide,(0,1))
print(four_slide)

five_slide = Kslide(four_slide,(0,0))
print(five_slide)

 Starting Tableau [[None, None, None, 2], [None, None, 2], [1, 3, 4]] and it's inner corners [(0, 2), (1, 1)] and outer corners [(0, 3), (2, 2)]
 intermediate tableau [[None, None, 2, None], [None, None, None], [1, 3, 4]] and the corner (1, 2)
 intermediate tableau [[None, None, 2, None], [None, None, 4], [1, 3, None]] and the corner (2, 2)
[[None, None, 2], [None, None, 4], [1, 3]]
 Starting Tableau [[None, None, 2], [None, None, 4], [1, 3]] and it's inner corners [(1, 1)] and outer corners [(1, 2), (2, 1)]
 intermediate tableau [[None, None, 2], [None, 3, 4], [1, None]]
[[None, None, 2], [None, 3, 4], [1]]
 Starting Tableau [[None, None, 2], [None, 3, 4], [1]] and it's inner corners [(0, 1), (1, 0)] and outer corners [(1, 2), (2, 0)]
 intermediate tableau [[None, None, 2], [1, 3, 4], [None]]
[[None, None, 2], [1, 3, 4]]
 Starting Tableau [[None, None, 2], [1, 3, 4]] and it's inner corners [(0, 1)] and outer corners [(1, 2)]
 intermediate tableau [[None, 2, None], [1, 3, 4]]
 intermed

In [50]:
def word_skewtab(word):
    n = len(word)
    tab = []
    k = n-1
    while k > -1:
        tab.append([None] * k + [word[k]])
        k -= 1
    return tab

def order_from_min_tableau(word):
    ## create skew tableaux (even tho t is a real tableau) to use the corners feature
    skewtab = SkewTableau(word_skewtab(word))
    outer_corners = skewtab.outer_shape().corners()
    #print(outer_corners)
    n = len(skewtab.to_list()[0])
    order_dic = {key: [] for key in range(1,n+1)}
    for (x,y) in outer_corners:
        order_dic[n].append((x,y))
        i = 1
        while x-i >= 0:
            order_dic[n-i].append((x-i,y))
            i += 1

    print(f" this is the order dict {order_dic}")
    order = order_dic[n-1]
    order_rev = list(order_dic[n-1])
    order_rev.reverse()
    #print(order_rev)
    i = n-2
    while i >= 1:
        order = order + order_dic[i]
        rev = list(order_dic[i])
        rev.reverse()
        order_rev = order_rev + rev
        i -= 1
        
    return (order, order_rev)
            

In [51]:
## first EG example
w = [6, 4, 1, 2, 5, 3, 4]

word_skewtab(w)

[[None, None, None, None, None, None, 4],
 [None, None, None, None, None, 3],
 [None, None, None, None, 5],
 [None, None, None, 2],
 [None, None, 1],
 [None, 4],
 [6]]

In [52]:
w = [6, 4, 1, 2, 5, 3, 4]
tab = SkewTableau(word_skewtab(w))
rect_order = order_from_min_tableau(w)[0]

Krect_from_slides(tab, rect_order)

 this is the order dict {1: [(0, 0)], 2: [(0, 1), (1, 0)], 3: [(0, 2), (1, 1), (2, 0)], 4: [(0, 3), (1, 2), (2, 1), (3, 0)], 5: [(0, 4), (1, 3), (2, 2), (3, 1), (4, 0)], 6: [(0, 5), (1, 4), (2, 3), (3, 2), (4, 1), (5, 0)], 7: [(0, 6), (1, 5), (2, 4), (3, 3), (4, 2), (5, 1), (6, 0)]}
 Starting Tableau [[None, None, None, None, None, None, 4], [None, None, None, None, None, 3], [None, None, None, None, 5], [None, None, None, 2], [None, None, 1], [None, 4], [6]] and it's inner corners [(0, 5), (1, 4), (2, 3), (3, 2), (4, 1), (5, 0)] and outer corners [(0, 6), (1, 5), (2, 4), (3, 3), (4, 2), (5, 1), (6, 0)]
 intermediate tableau [[None, None, None, None, None, 3, 4], [None, None, None, None, None, None], [None, None, None, None, 5], [None, None, None, 2], [None, None, 1], [None, 4], [6]]
 Starting Tableau [[None, None, None, None, None, 3, 4], [None, None, None, None, None], [None, None, None, None, 5], [None, None, None, 2], [None, None, 1], [None, 4], [6]] and it's inner corners [(1, 4),

[[1, 2, 3, 4], [4, 5], [6]]

In [53]:
RSK(w, insertion=RSK.rules.EG)

[[[1, 2, 3, 4], [4, 5], [6]], [[1, 4, 5, 7], [2, 6], [3]]]

In [76]:
w = [4,2,1,2,3,2,4]
tab = SkewTableau(word_skewtab(w))
rect_order = order_from_min_tableau(w)[0]

T = Krect_from_slides(tab, rect_order)
RSK(w, insertion=RSK.rules.EG)

 this is the order dict {1: [(0, 0)], 2: [(0, 1), (1, 0)], 3: [(0, 2), (1, 1), (2, 0)], 4: [(0, 3), (1, 2), (2, 1), (3, 0)], 5: [(0, 4), (1, 3), (2, 2), (3, 1), (4, 0)], 6: [(0, 5), (1, 4), (2, 3), (3, 2), (4, 1), (5, 0)], 7: [(0, 6), (1, 5), (2, 4), (3, 3), (4, 2), (5, 1), (6, 0)]}
 here's the starting corner to slide (0, 5)
 Starting Tableau [[None, None, None, None, None, None, 4], [None, None, None, None, None, 2], [None, None, None, None, 3], [None, None, None, 2], [None, None, 1], [None, 2], [4]] and it's inner corners [(0, 5), (1, 4), (2, 3), (3, 2), (4, 1), (5, 0)] and outer corners [(0, 6), (1, 5), (2, 4), (3, 3), (4, 2), (5, 1), (6, 0)]
 intermediate tableau [[None, None, None, None, None, 2, 4], [None, None, None, None, None, None], [None, None, None, None, 3], [None, None, None, 2], [None, None, 1], [None, 2], [4]]
 Starting Tableau [[None, None, None, None, None, 2, 4], [None, None, None, None, None], [None, None, None, None, 3], [None, None, None, 2], [None, None, 1], [No

[[[1, 2, 3, 4], [2, 3], [4]], [[1, 4, 5, 7], [2, 6], [3]]]

In [63]:
w_intermed = [4,2,1,2,3,2,3]
tab = SkewTableau(word_skewtab(w_intermed))
rect_order = order_from_min_tableau(w_intermed)[0]

print(Krect_from_slides(tab, rect_order))
RSK(w_intermed, insertion=RSK.rules.EG)

 this is the order dict {1: [(0, 0)], 2: [(0, 1), (1, 0)], 3: [(0, 2), (1, 1), (2, 0)], 4: [(0, 3), (1, 2), (2, 1), (3, 0)], 5: [(0, 4), (1, 3), (2, 2), (3, 1), (4, 0)], 6: [(0, 5), (1, 4), (2, 3), (3, 2), (4, 1), (5, 0)], 7: [(0, 6), (1, 5), (2, 4), (3, 3), (4, 2), (5, 1), (6, 0)]}
 here's the corner to slide (0, 5)
 Starting Tableau [[None, None, None, None, None, None, 3], [None, None, None, None, None, 2], [None, None, None, None, 3], [None, None, None, 2], [None, None, 1], [None, 2], [4]] and it's inner corners [(0, 5), (1, 4), (2, 3), (3, 2), (4, 1), (5, 0)] and outer corners [(0, 6), (1, 5), (2, 4), (3, 3), (4, 2), (5, 1), (6, 0)]
 intermediate tableau [[None, None, None, None, None, 2, 3], [None, None, None, None, None, None], [None, None, None, None, 3], [None, None, None, 2], [None, None, 1], [None, 2], [4]]
 here's the corner to slide (1, 4)
 Starting Tableau [[None, None, None, None, None, 2, 3], [None, None, None, None, None], [None, None, None, None, 3], [None, None, None

[[[1, 2, 3, 3], [2, 3], [4]], [[1, 4, 5, 7], [2, 6], [3]]]

In [79]:
w_intermed = [4,2,1,1,3,2,3]
tab = SkewTableau(word_skewtab(w_intermed))
rect_order = order_from_min_tableau(w_intermed)[0]

print(Krect_from_slides(tab, rect_order))
RSK(w_intermed, insertion=RSK.rules.EG)

 this is the order dict {1: [(0, 0)], 2: [(0, 1), (1, 0)], 3: [(0, 2), (1, 1), (2, 0)], 4: [(0, 3), (1, 2), (2, 1), (3, 0)], 5: [(0, 4), (1, 3), (2, 2), (3, 1), (4, 0)], 6: [(0, 5), (1, 4), (2, 3), (3, 2), (4, 1), (5, 0)], 7: [(0, 6), (1, 5), (2, 4), (3, 3), (4, 2), (5, 1), (6, 0)]}
 here's the starting corner to slide (0, 5)
 Starting Tableau [[None, None, None, None, None, None, 3], [None, None, None, None, None, 2], [None, None, None, None, 3], [None, None, None, 1], [None, None, 1], [None, 2], [4]] and it's inner corners [(0, 5), (1, 4), (2, 3), (3, 2), (4, 1), (5, 0)] and outer corners [(0, 6), (1, 5), (2, 4), (3, 3), (4, 2), (5, 1), (6, 0)]
 intermediate tableau [[None, None, None, None, None, 2, 3], [None, None, None, None, None, None], [None, None, None, None, 3], [None, None, None, 1], [None, None, 1], [None, 2], [4]]
 Starting Tableau [[None, None, None, None, None, 2, 3], [None, None, None, None, None], [None, None, None, None, 3], [None, None, None, 1], [None, None, 1], [No

[[[1, 1, 2, 3], [2, 3], [4]], [[1, 4, 5, 7], [2, 6], [3]]]

In [193]:
w_intermed = [5,3,1,2,4,3,4]
tab = SkewTableau(word_skewtab(w_intermed))
rect_order = order_from_min_tableau(w_intermed)[0]

print(Krect_from_slides(tab, rect_order))
RSK(w_intermed, insertion=RSK.rules.EG)

[(0, 6), (1, 5), (2, 4), (3, 3), (4, 2), (5, 1), (6, 0)]
 this is the order dict {1: [(0, 0)], 2: [(0, 1), (1, 0)], 3: [(0, 2), (1, 1), (2, 0)], 4: [(0, 3), (1, 2), (2, 1), (3, 0)], 5: [(0, 4), (1, 3), (2, 2), (3, 1), (4, 0)], 6: [(0, 5), (1, 4), (2, 3), (3, 2), (4, 1), (5, 0)], 7: [(0, 6), (1, 5), (2, 4), (3, 3), (4, 2), (5, 1), (6, 0)]}
[(5, 0), (4, 1), (3, 2), (2, 3), (1, 4), (0, 5)]
 here's [[None, None, None, None, None, None, 4], [None, None, None, None, None, 3], [None, None, None, None, 4], [None, None, None, 2], [None, None, 1], [None, 3], [5]] and [(0, 5), (1, 4), (2, 3), (3, 2), (4, 1), (5, 0)] and [(0, 6), (1, 5), (2, 4), (3, 3), (4, 2), (5, 1), (6, 0)]
case compare
swap below
 here's the intermediate tableau [[None, None, None, None, None, 3, 4], [None, None, None, None, None, None], [None, None, None, None, 4], [None, None, None, 2], [None, None, 1], [None, 3], [5]]
 here's [[None, None, None, None, None, 3, 4], [None, None, None, None, None], [None, None, None, None, 4],

[[[1, 2, 3, 4], [3, 4], [5]], [[1, 4, 5, 7], [2, 6], [3]]]

In [40]:
w_intermed = [5,3,1,2,4,3,4]
tab = SkewTableau(word_skewtab(w_intermed))
rect_order = order_from_min_tableau(w_intermed)[1]

print(Krect_from_slides(tab, rect_order))
RSK(w_intermed, insertion=RSK.rules.EG)

[(0, 6), (1, 5), (2, 4), (3, 3), (4, 2), (5, 1), (6, 0)]
 this is the order dict {1: [(0, 0)], 2: [(0, 1), (1, 0)], 3: [(0, 2), (1, 1), (2, 0)], 4: [(0, 3), (1, 2), (2, 1), (3, 0)], 5: [(0, 4), (1, 3), (2, 2), (3, 1), (4, 0)], 6: [(0, 5), (1, 4), (2, 3), (3, 2), (4, 1), (5, 0)], 7: [(0, 6), (1, 5), (2, 4), (3, 3), (4, 2), (5, 1), (6, 0)]}
[(5, 0), (4, 1), (3, 2), (2, 3), (1, 4), (0, 5)]
 Starting Tableau [[None, None, None, None, None, None, 4], [None, None, None, None, None, 3], [None, None, None, None, 4], [None, None, None, 2], [None, None, 1], [None, 3], [5]] and it's inner corners [(0, 5), (1, 4), (2, 3), (3, 2), (4, 1), (5, 0)] and outer corners [(0, 6), (1, 5), (2, 4), (3, 3), (4, 2), (5, 1), (6, 0)]
 intermediate tableau [[None, None, None, None, None, None, 4], [None, None, None, None, None, 3], [None, None, None, None, 4], [None, None, None, 2], [None, None, 1], [3, None], [5]]
 Starting Tableau [[None, None, None, None, None, None, 4], [None, None, None, None, None, 3], [Non

[[[1, 2, 3, 4], [3, 4], [5]], [[1, 4, 5, 7], [2, 6], [3]]]

In [41]:
v = [3,1,1,3]
tab = SkewTableau(word_skewtab(v))
rect_order = order_from_min_tableau(v)[1]

print(Krect_from_slides(tab, rect_order))
RSK(v, insertion=RSK.rules.Hecke)

[(0, 3), (1, 2), (2, 1), (3, 0)]
 this is the order dict {1: [(0, 0)], 2: [(0, 1), (1, 0)], 3: [(0, 2), (1, 1), (2, 0)], 4: [(0, 3), (1, 2), (2, 1), (3, 0)]}
[(2, 0), (1, 1), (0, 2)]
 Starting Tableau [[None, None, None, 3], [None, None, 1], [None, 1], [3]] and it's inner corners [(0, 2), (1, 1), (2, 0)] and outer corners [(0, 3), (1, 2), (2, 1), (3, 0)]
 intermediate tableau [[None, None, None, 3], [None, None, 1], [1, None], [3]]
 Starting Tableau [[None, None, None, 3], [None, None, 1], [1], [3]] and it's inner corners [(0, 2), (1, 1)] and outer corners [(0, 3), (1, 2), (3, 0)]
 intermediate tableau [[None, None, None, 3], [None, 1, None], [1], [3]] and the corner (1, 2)
 Starting Tableau [[None, None, None, 3], [None, 1], [1], [3]] and it's inner corners [(0, 2), (1, 0)] and outer corners [(0, 3), (1, 1), (3, 0)]
 intermediate tableau [[None, None, 3, None], [None, 1], [1], [3]] and the corner (0, 3)
 Starting Tableau [[None, None, 3], [None, 1], [1], [3]] and it's inner corners [(

[[[1, 3], [3]], [[(1,), (4,)], [(2, 3)]]]